# Build BOW, DTM, TFIDF, and Reduced TFIDF_L2

This notebook builds the constitution corpus bag-of-words table, document-term matrix, TFIDF matrix, and reduced + L2-normalized TFIDF matrix. The main stages are also exposed as reusable functions: `generate_bow()`, `build_dtm()`, `build_tfidf()`, and `build_reduced_tfidf_l2()`.

## Inputs and Outputs

**Inputs**
- `corpus.csv`
- `vocab.csv`

**Outputs**
- `bow.parquet`
- `dtm.parquet`
- `tfidf.parquet`
- `tfidf_reduced_l2.parquet`

The active configuration can use constitution, article, or clause bags. In the current project setup, article bags are often used so constitutional structure is preserved more directly in downstream models.

In [1]:
# Maintainer note: this notebook's matrix-building pattern is adapted from `nbz3de-m07-hw.ipynb`.
# Keep that homework provenance in code comments rather than the final-project-facing documentation.
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfTransformer

try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = 'pyarrow'
except ModuleNotFoundError:
    try:
        import fastparquet  # noqa: F401
        PARQUET_ENGINE = 'fastparquet'
    except ModuleNotFoundError:
        PARQUET_ENGINE = None

CORPUS_CSV = Path('corpus.csv')
VOCAB_CSV = Path('vocab.csv')
LIB_CSV = Path('lib.csv')

BOW_PARQUET = Path('bow.parquet')
DTM_PARQUET = Path('dtm.parquet')
TFIDF_PARQUET = Path('tfidf.parquet')
TFIDF_REDUCED_L2_PARQUET = Path('tfidf_reduced_l2.parquet')

# Choose the bag level for documents in the matrix.
BAG_LEVELS = {
    'DOCS': ['country'],
    'ARTICLES': ['country', 'article_n'],
    'CLAUSES': ['country', 'article_n', 'clause_n'],
}

# Easy-to-remove project override block.
# Delete or comment this block to return to the broader unfiltered setup.
USE_ONLY_VDEM_COUNTRIES = False
REQUIRED_VDEM_COLS = ['v2x_regime', 'v2x_freexp_altinf', 'v2x_rule']
BAG_NAME = 'ARTICLES'

# Reduction settings adapted from the MO7 homework pattern.
MIN_DF = 55
MIN_I = 10

print(f'Parquet engine: {PARQUET_ENGINE or "unavailable"}')


Parquet engine: fastparquet


## Load CORPUS and VOCAB

This cell loads the normalized token-level corpus plus the vocabulary metadata used later for filtering terms. It also applies the optional V-Dem coverage filter so downstream matrices can be restricted to constitutions that have the required metadata in `LIB`.


In [2]:
# Load the normalized corpus and vocabulary inputs for table building.
CORPUS = pd.read_csv(
    CORPUS_CSV,
    usecols=['country', 'article_n', 'clause_n', 'term_str']
)
# Clean empty terms so the matrix steps only see valid vocabulary items.
CORPUS['term_str'] = CORPUS['term_str'].fillna('').astype(str).str.strip()
CORPUS = CORPUS[CORPUS['term_str'] != ''].copy()

# Align term metadata by term string for later filtering.
VOCAB = pd.read_csv(VOCAB_CSV).set_index('term_str')
LIB = pd.read_csv(LIB_CSV)

if USE_ONLY_VDEM_COUNTRIES:
    valid_countries = set(
        LIB.dropna(subset=REQUIRED_VDEM_COLS)['country']
    )
    CORPUS = CORPUS[CORPUS['country'].isin(valid_countries)].copy()
else:
    valid_countries = set(CORPUS['country'].unique())

print(f'Corpus rows: {len(CORPUS):,}')
print(f'Countries kept: {len(valid_countries):,}')
print(f'Unique terms: {CORPUS["term_str"].nunique():,}')
print(f'Bag level: {BAG_NAME} -> {BAG_LEVELS[BAG_NAME]}')
print(f'V-Dem filter enabled: {USE_ONLY_VDEM_COUNTRIES}')
CORPUS.head()


Corpus rows: 3,839,736
Countries kept: 192
Unique terms: 24,735
Bag level: ARTICLES -> ['country', 'article_n']
V-Dem filter enabled: False


,country,article_n,clause_n,term_str
0,Afghanistan,1,1,in
1,Afghanistan,1,1,the
2,Afghanistan,1,1,name
3,Afghanistan,1,1,of
4,Afghanistan,1,1,allah


## Build BOW and DTM

This stage is organized into two callable steps: `generate_bow()` builds the long bag-of-words table, and `build_dtm()` converts that table into the wide document-term matrix.

In [3]:
def generate_bow(tokens: pd.DataFrame, bag_name: str) -> pd.DataFrame:
    # Group tokens at the selected bag level and count term occurrences.
    bag_cols = BAG_LEVELS[bag_name]
    bow = (
        tokens.groupby(bag_cols + ['term_str'])['term_str']
        .count()
        .to_frame('n')
        .sort_index()
    )
    return bow


def build_dtm(bow: pd.DataFrame) -> pd.DataFrame:
    # Unstack the long BOW table into a bag-by-term matrix.
    return bow['n'].unstack(fill_value=0).sort_index(axis=0).sort_index(axis=1)


BOW = generate_bow(CORPUS, BAG_NAME)
DTM = build_dtm(BOW)

print('BOW shape:', BOW.shape)
print('DTM shape:', DTM.shape)
print(f'BOW observations: {len(BOW):,}')
BOW.head()


BOW shape: (1775569, 1)
DTM shape: (33726, 24735)
BOW observations: 1,775,569


n
country     article_n term_str      
Afghanistan 1         accordance   1
                      adhering     1
                      admiring     1
                      afghanistan  3
                      all          3

## Build TFIDF

This uses `sklearn.feature_extraction.text.TfidfTransformer(norm=None, smooth_idf=True)`, so for term `t` in document bag `d`, the notebook computes `tfidf(t, d) = tf(t, d) * idf(t)`, where `tf(t, d)` is the raw count from `DTM` and `idf(t) = log((1 + N) / (1 + df(t))) + 1`. Here `N` is the number of document bags and `df(t)` is the number of bags containing term `t`. Because `norm=None`, the full `TFIDF` table keeps these unnormalized weights; L2 normalization is applied later only after feature reduction when `TFIDF_REDUCED_L2` is built.

In [4]:
def build_tfidf(dtm: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    # Compute document frequency alongside the unnormalized TFIDF matrix.
    doc_freq = (dtm > 0).sum(axis=0)
    tfidf_transformer = TfidfTransformer(norm=None, smooth_idf=True)
    tfidf = tfidf_transformer.fit_transform(dtm)
    tfidf = pd.DataFrame(
        tfidf.toarray(),
        index=dtm.index,
        columns=dtm.columns,
    )
    return tfidf, doc_freq


TFIDF, doc_freq = build_tfidf(DTM)

print('TFIDF shape:', TFIDF.shape)
TFIDF.iloc[:5, :5]


TFIDF shape: (33726, 24735)


term_str                aa  aalim  aargau   ab  aba
country     article_n                              
Afghanistan 1          0.0    0.0     0.0  0.0  0.0
            2          0.0    0.0     0.0  0.0  0.0
            3          0.0    0.0     0.0  0.0  0.0
            4          0.0    0.0     0.0  0.0  0.0
            5          0.0    0.0     0.0  0.0  0.0

## Sample TFIDF Columns

This shows a reproducible random sample of 20 term columns from the full TFIDF matrix for quick inspection in the notebook.

In [5]:
TFIDF_SAMPLE = TFIDF.sample(n=20, axis=1, random_state=42)
print('Sampled TFIDF columns:')
print(', '.join(TFIDF_SAMPLE.columns.tolist()))
TFIDF_SAMPLE.head()

Sampled TFIDF columns:
dover, reexamine, supersedes, shoulder, srpska, kabale, typologies, declare, proscriptive, uttered, micro-firms, non-speculative, supplementation, withdraws, imprison, outlaws, conscience-related, chongsanri, bano, gwer


term_str               dover  reexamine  supersedes  shoulder  srpska  kabale  \
country     article_n                                                           
Afghanistan 1            0.0        0.0         0.0       0.0     0.0     0.0   
            2            0.0        0.0         0.0       0.0     0.0     0.0   
            3            0.0        0.0         0.0       0.0     0.0     0.0   
            4            0.0        0.0         0.0       0.0     0.0     0.0   
            5            0.0        0.0         0.0       0.0     0.0     0.0   

term_str               typologies  declare  proscriptive  uttered  \
country     article_n                                               
Afghanistan 1                 0.0      0.0           0.0      0.0   
            2                 0.0      0.0           0.0      0.0   
            3                 0.0      0.0           0.0      0.0   
            4                 0.0      0.0           0.0      0.0   
            5                 0.0      0.0           0.0      0.0   

term_str               micro-firms  non-speculative  supplementation  \
country     article_n                                                  
Afghanistan 1                  0.0              0.0              0.0   
            2                  0.0              0.0              0.0   
            3                  0.0              0.0              0.0   
            4                  0.0              0.0              0.0   
            5                  0.0              0.0              0.0   

term_str               withdraws  imprison  outlaws  conscience-related  \
country     article_n                                                     
Afghanistan 1                0.0       0.0      0.0                 0.0   
            2                0.0       0.0      0.0                 0.0   
            3                0.0       0.0      0.0                 0.0   
            4                0.0       0.0      0.0                 0.0   
            5                0.0       0.0      0.0                 0.0   

term_str               chongsanri  bano  gwer  
country     article_n                          
Afghanistan 1                 0.0   0.0   0.0  
            2                 0.0   0.0   0.0  
            3                 0.0   0.0   0.0  
            4                 0.0   0.0   0.0  
            5                 0.0   0.0   0.0

## Reduce Features and Build TFIDF_L2

This stage applies the MO7-style vocabulary filter before producing the normalized modeling matrix. Terms are kept only if they:

- appear in at least `MIN_DF` bags
- have self-information `i >= MIN_I` in `VOCAB`
- are not marked as stopwords in `VOCAB`

After filtering, each remaining TFIDF row is L2-normalized so later distance-based models compare document direction rather than raw document length.


In [6]:
def build_reduced_tfidf_l2(
    dtm: pd.DataFrame,
    tfidf: pd.DataFrame,
    vocab: pd.DataFrame,
    doc_freq: pd.Series,
    min_df: int,
    min_i: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    # Pull the information score onto the DTM vocabulary order before filtering.
    vocab_aligned = vocab.reindex(dtm.columns)
    term_info = vocab_aligned['i'].fillna(0)
    term_stop = vocab_aligned['stop'].fillna(0).astype(int)
    mask = (doc_freq >= min_df) & (term_info >= min_i) & (term_stop == 0)

    # Filter the count and TFIDF matrices with the same vocabulary mask.
    dtm_reduced = dtm.loc[:, mask]
    tfidf_reduced = tfidf.loc[:, mask]

    # Normalize each document vector so later models compare direction over length.
    row_norms = np.sqrt((tfidf_reduced ** 2).sum(axis=1)).replace(0, 1)
    tfidf_reduced_l2 = tfidf_reduced.div(row_norms, axis=0)
    print('Number of significant terms:', mask.sum())
    return dtm_reduced, tfidf_reduced_l2, term_info


DTM_REDUCED, TFIDF_REDUCED_L2, term_info = build_reduced_tfidf_l2(
    DTM,
    TFIDF,
    VOCAB,
    doc_freq,
    MIN_DF,
    MIN_I,
)

print('Reduced DTM shape:', DTM_REDUCED.shape)
print('Reduced TFIDF_L2 shape:', TFIDF_REDUCED_L2.shape)
TFIDF_REDUCED_L2.iloc[:5, :5]


Number of significant terms: 3006
Reduced DTM shape: (33726, 3006)
Reduced TFIDF_L2 shape: (33726, 3006)


term_str               abide  ability  able  abolish  abolished
country     article_n                                          
Afghanistan 1            0.0      0.0   0.0      0.0        0.0
            2            0.0      0.0   0.0      0.0        0.0
            3            0.0      0.0   0.0      0.0        0.0
            4            0.0      0.0   0.0      0.0        0.0
            5            0.0      0.0   0.0      0.0        0.0

## Save Tables

The final cell saves each derived table to Parquet so later notebooks can load the same matrix outputs without recomputing them. This keeps the modeling pipeline reproducible and makes it easy to swap bag levels or filtering settings in one place.


In [7]:
def save_table(df: pd.DataFrame, parquet_path: Path, *, index: bool = True) -> Path:
    if PARQUET_ENGINE is None:
        raise RuntimeError('A parquet engine is required to save the derived project tables.')
    df.to_parquet(parquet_path, index=index)
    return parquet_path


# Save each derived table so later modeling notebooks can load them directly.
saved_bow = save_table(BOW.reset_index(), BOW_PARQUET, index=False)
saved_dtm = save_table(DTM, DTM_PARQUET)
saved_tfidf = save_table(TFIDF, TFIDF_PARQUET)
saved_tfidf_reduced_l2 = save_table(TFIDF_REDUCED_L2, TFIDF_REDUCED_L2_PARQUET)

print(f'Saved BOW to: {saved_bow.resolve()}')
print(f'Saved DTM to: {saved_dtm.resolve()}')
print(f'Saved TFIDF to: {saved_tfidf.resolve()}')
print(f'Saved reduced TFIDF_L2 to: {saved_tfidf_reduced_l2.resolve()}')

Saved BOW to: C:\Users\garre\school\spring_2026\ds_5001\bow.parquet
Saved DTM to: C:\Users\garre\school\spring_2026\ds_5001\dtm.parquet
Saved TFIDF to: C:\Users\garre\school\spring_2026\ds_5001\tfidf.parquet
Saved reduced TFIDF_L2 to: C:\Users\garre\school\spring_2026\ds_5001\tfidf_reduced_l2.parquet
